In [30]:
"""
customer support agent that can pull answers from two fundamentally different data sources in a single conversation:

1) Structured data (SQLite) - Customer records, order history, and product inventory stored in relational tables. 
The agent queries these through parameterized functions (never raw SQL).
2) Unstructured data (Knowledge Base) - Markdown documents covering return policies, shipping info, FAQs, and pricing plans. 
The agent searches these by keyword.

    Tool	                Data Source	                                    Purpose
--------------            -------------------                   ---------------------------------------
search_orders	            SQLite orders + customers	        Look up orders by email, ID, or status
search_products	            SQLite products	                    Find products by category, keyword, or stock 

The key insight is that the model never generates SQL directly. 
Each tool accepts structured parameters (email, order ID, category, etc.) and the Python function constructs the appropriate 
query internally. This is safer and more predictable than text-to-SQL approaches.
"""

'\ncustomer support agent that can pull answers from two fundamentally different data sources in a single conversation:\n\n1) Structured data (SQLite) - Customer records, order history, and product inventory stored in relational tables. \nThe agent queries these through parameterized functions (never raw SQL).\n2) Unstructured data (Knowledge Base) - Markdown documents covering return policies, shipping info, FAQs, and pricing plans. \nThe agent searches these by keyword.\n\n    Tool\t                Data Source\t                                    Purpose\n--------------            -------------------                   ---------------------------------------\nsearch_orders\t            SQLite orders + customers\t        Look up orders by email, ID, or status\nsearch_products\t            SQLite products\t                    Find products by category, keyword, or stock \n\nThe key insight is that the model never generates SQL directly. \nEach tool accepts structured parameters (email, 

In [31]:
import os
from getpass import getpass
import json
import sqlite3
import pathlib
from typing import Literal
import sys

from llm_config import ollama, MODEL_OLLAMA
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

In [32]:
# Sample data (JSON): customers, products, and orders
# Knowledge base (Markdown files): return policy, shipping info, FAQ, pricing
# System instructions (text file): the prompt that tells the agent how to behave

In [33]:
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")
RESOURCES = pathlib.Path("..","resources","knowledge_base")
data = json.loads((RESOURCES / "data" / "customer_support_sample.json").read_text())
print(f"Loaded {len(data['customers'])} customers, "
      f"{len(data['products'])} products, "
      f"{len(data['orders'])} orders")

 
knowledge_base = {}
for md_file in sorted((RESOURCES).glob("*.md")):     # Find every file whose name ends with .md.
    knowledge_base[md_file.name] = md_file.read_text()                           # uses the filename as the dictionary key and the file contents as the value.
print(f"\nKnowledge base contains {len(knowledge_base)} documents:")
for filename, content in knowledge_base.items():
    line_count = len(content.strip().splitlines())
    print(f"  - {filename} ({line_count} lines)")

"""
knowledge_base = {
    "payments.md": "Payment information....................data from file as a value of the key",
    "returns.md": "Returns Policy\n\nCustomers can return................data from file as a value of the key",
    "shipping.md": "Shipping information...data from file as a value of the key"
    
    Markdown files
        ↓
    Read files
        ↓
    Python dictionary
        ↓
    AI Agent / RAG
}
"""

instructions = (RESOURCES / "prompts" / "customer_support_agent_instructions.txt").read_text()
print(f"\nSystem instructions ({len(instructions)} chars):")
print(instructions[:200])


Loaded 8 customers, 10 products, 10 orders

Knowledge base contains 4 documents:
  - faq.md (16 lines)
  - pricing_plans.md (22 lines)
  - return_policy.md (19 lines)
  - shipping_info.md (17 lines)

System instructions (314 chars):
You are a helpful customer support agent for an online store.
Use the available tools to look up information before answering.
Always cite which source you used (order database, product catalog, or kn


In [34]:
# Now let's create an in-memory SQLite database and populate it with the data we just loaded. Notice how the table schemas enforce constraints (valid plan types, valid order statuses) to keep the data clean

# Create a temporary database → make query results easy to access by column name → create a cursor for running SQL commands.
# A cursor is what you use to send SQL commands to the database.
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # Return rows as dictionaries
cursor = conn.cursor()

cursor.executescript("""                        
CREATE TABLE customers (                                
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    plan_type TEXT NOT NULL CHECK(plan_type IN ('free', 'pro', 'enterprise')),
    created_at TEXT NOT NULL
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    in_stock INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    total_price REAL NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('pending', 'shipped', 'delivered', 'refunded')),
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);
""")

# Populate from loaded JSON data
for c in data["customers"]:
    cursor.execute(
        "INSERT INTO customers VALUES (?, ?, ?, ?, ?)",
        (c["id"], c["name"], c["email"], c["plan_type"], c["created_at"]),
    )

for p in data["products"]:
    cursor.execute(
        "INSERT INTO products VALUES (?, ?, ?, ?, ?)",
        (p["id"], p["name"], p["category"], p["price"], p["in_stock"]),
    )

for o in data["orders"]:
    cursor.execute(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)",
        (o["id"], o["customer_id"], o["product_name"],
         o["quantity"], o["total_price"], o["status"], o["created_at"]),
    )

conn.commit()

# Verify the data
for table in ["customers", "products", "orders"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")


customers: 8 rows
products: 10 rows
orders: 10 rows


In [35]:
# Define Pydantic Tool Argument Models
# Instead of hand-typing JSON schemas for each tool (which is tedious and error-prone), 
# we define Pydantic models that describe the arguments for each tool. 
# Pydantic can then auto-generate the exact JSON schema that OpenAI expects.
from pydantic import BaseModel, ConfigDict, Field

class ToolArgs(BaseModel):
    ConfigDict(extra = "forbid")

def tool_schema_function(name:str, description:str, tool_model: type[BaseModel]) -> dict:
    return {
        "type" : "function",
        "name" : name,
        "description" : description,
        "parameters" : tool_model.model_json_schema()
    }
class SearchCustomers(BaseModel):
    cust_id : int
    location : str

class SearchOrdersArgs(ToolArgs):
    customer_email: str | None = Field(None, description='customer email address to look up orders for.')   # customer_email can contain a string or nothing.
    order_id: int | None = Field(None, description='Specific order ID to look up')
    status: Literal['pending', 'shipped', 'delivered', 'refunded'] = Field(None, description='Filter orders by status')  # Field() provides extra information about the input.

class SearchProductsArgs(ToolArgs):
    category: str | None = Field(None, description='Product category (e.g. Electronics, Accessories, Furniture)')
    search_term: str | None = Field(None, description='Keyword to search for in product names')
    in_stock_only: bool = Field(False, description='If true, only return products currently in stock')


# tool_schema_function("Jim", "Retail customer from AZ", SearchCustomers)

tools = [
    tool_schema_function("search_orders", "Search for customer orders by email, order ID, or status.", SearchOrdersArgs),
    tool_schema_function("search_products", "Search the product catalog by category, name, or stock.", SearchProductsArgs), 
]


"""
name: str
  │     │
  │     └── name must be text 
  └── variable name
= Field(...)
  │
  └── additional information/rules
"""



'\nname: str\n  │     │\n  │     └── name must be text \n  └── variable name\n= Field(...)\n  │\n  └── additional information/rules\n'

In [36]:
# Build functions . For example search_orders
# Each tool function accepts structured parameters and internally constructs the appropriate SQL query or search logic. 
# The model never sees or generates raw SQL; it simply passes parameters like customer_email or category, 
# and the function handles the rest.

In [37]:
def search_orders(**kwargs):
    args = SearchOrdersArgs(**kwargs)
    query = """                                  # create multi-line string and put all that into query variable 
        SELECT 
            o.id AS order_id,
            c.name as customer_name,
            c.email,
            o.product_name,
            o.quantity,
            o.total_price,
            o.status,
            o.created_at
        FROM orders o
        JOIN customers c ON o.customer_id = c.id
        WHERE 1=1
    """
    params = []
    if args.customer_email:
        query += " AND c.email = ?"
        params.append(args.customer_email)
    if args.order_id is not None:
        query += " AND o.id = ?"
        params.append(args.order_id)
    if args.status:
        query += " AND o.status = ?"
        params.append(args.status)
    query += ' ORDER BY o.created_at DESC'

    rows = cursor.execute(query, params).fetchall()            # Execute the query:
    results = [dict(row)for row in rows]
    if not results:
        return json.dumps({"message": "No orders found matching the criteria."})
    return json.dumps(results, indent=2)

In [38]:
# Tool Dispatch - When the model calls a tool, we need to route that call to the right Python function. 
# The dispatcher below handles argument parsing and filters out None values so the function defaults kick in properly.
# LLM says which tool it wants → this code finds that tool → passes the arguments → runs it → returns the result.

# Tool to Function mapping 
TOOLS_FUNCTIONS = {
    "search_orders": search_orders,
#    "search_products": search_products,
}

def dispatch_tool_call(name: str, args: dict):
    func = TOOLS_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        filtered_args = {k: v for k, v in args.items() if v is not None}
        return func(**filtered_args)
    except Exception as e:
        return json.dumps({"error": str(e)})

In [49]:
# The Agent Loop
"""
This is the heart of our agent. The loop follows a simple pattern:

    1. Send the user's question to the model along with the available tools.
    2 .If the model returns a text response, we are done; that is the final answer.
    3. If the model returns one or more tool calls, execute each one and feed the results back.
    4. Repeat until the model produces a final text answer or we hit the turn limit.
This loop allows the model to chain multiple tool calls together. 
For example, it might first search for a customer's orders, then look up the return policy, and 
finally synthesize both pieces of information into a single answer.
"""
def run_agent(user_query, max_turns=10):
    messages = [{"role": "user", "content": user_query}]
    number_of_turns = 0
    
    while number_of_turns <= max_turns:
        response = ollama.responses.create(
            model=MODEL_OLLAMA,
            instructions=instructions,
            input=messages,
            tools=tools
        )

        number_of_turns += 1
        print(f" Number of turns: {number_of_turns}")

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            print("We finished here as no tool calls left to make")
            return response.output_text

        # Append entire response.output (reasoning + function_calls) - required for reasoning models
        messages.extend(response.output)

        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)
            print(args)
            result = dispatch_tool_call(tool_call.name, args)
            print(result)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result
            })
    return response.output_text

In [50]:
query = "What is the status of order 1003?"
run_agent(query)

 Number of turns: 1
{'status': 'pending', 'order_id': '1003'}
{"error": "unrecognized token: \"#\""}
 Number of turns: 2
We finished here as no tool calls left to make


"I was unable to find any information on the status of order 1003. It's possible that the order is no longer active or has been cancelled. To get the most up-to-date information, I recommend checking with our customer service team directly.\n\nYou can reach us by contacting our support team via phone or email, and we'll be happy to assist you in getting an update on your order status. Would you like me to provide your contact information?"